In [ ]:
import geopandas as gpd
import pandas as pd
import seaborn as sns

# Part 1

In [ ]:
from shapely import from_wkt

def combine_districts_and_census_areas(df, district_column='codi_districte', census_column='codi_seccio_censal'):
    """Combines districts and census areas into a single column, `Seccio_Censal`."""
    df["Seccio_Censal"] = (
        (df[district_column] * 1000 ) + df[census_column]
    )

def barcelona_census_areas(url="BarcelonaCiutat_SeccionsCensals.csv"):
    """This loads the census areas as defined in a CSV file, and returns a 
    GDF with a single `Seccio_Censal` column and associated geometry.
    
    This CSV file is taken from 
    https://opendata-ajuntament.barcelona.cat/data/en/dataset/20170706-districtes-barris/resource/e16856a7-b3c0-4c32-a468-cc190cbbf7a9
    where it is described as "Detail of administrative units of the city of Barcelona : districts, neighbourhoods, Interés area, basic statistical areas (AEB) and census areas"

    The `Seccio_Censal` column is the combination of the `codi_districte` and `codi_seccio_censal`. Many other datasets
    use this as the join key and it appears to be the combination of the district and census area columns.
    
    """
    url = "BarcelonaCiutat_SeccionsCensals.csv"
    df = pd.read_csv(url)
    combine_districts_and_census_areas(df)
    
    df["geometry"] = df["geometria_wgs84"].apply(from_wkt)
    df = df[["Seccio_Censal","geometry"]]
    assert df["Seccio_Censal"].is_unique
    df.set_index("Seccio_Censal")
    return gpd.GeoDataFrame(df,geometry="geometry",crs="EPSG:4326")


In [ ]:
census_areas_gdf = barcelona_census_areas()

In [ ]:
census_areas_gdf

In [ ]:
census_areas_gdf.explore("Seccio_Censal")

In [ ]:
def std_dev(df, column, std_column=None):
    std_df = df.copy(deep=True)
    if std_column is None:
        std_column = f"{column}_std"
    
    std_df[std_column] = (
        std_df[column] - std_df[column].mean()
    ) / std_df[column].std()

    return std_df

In [ ]:
def filter_extremes(df, column, max_abs_std=3):
    """filters out rows which are more than `max_abs_std` deviations away from zero."""
    std_column = f"{column}_std"
    std_df = std_dev(df, column, std_column = std_column)
    std_df = std_df[
        (std_df[std_column] >= (-1 * max_abs_std))
        & (std_df[std_column] <= max_abs_std)
    ]
    return std_df.drop(std_column, axis=1)

In [ ]:
import tobler
from libpysal import graph

def build_safe_grid(gdf, resolution):
    """Builds a safe H3 grid, where safe means it has no isolates.

    Some later operations don't work if we have isolates (areas with no nieghhours), so we exclude them here.
    """
    initial_grid_gdf = tobler.util.h3fy(gdf, resolution=resolution)
    
    # build a Queen contiguity and use it to exclude the isolates from the grid
    contiguity = graph.Graph.build_contiguity(initial_grid_gdf, rook=False)
    isolates = contiguity.isolates
    filtered_grid_gdf = initial_grid_gdf[~initial_grid_gdf.index.isin(isolates)]
    
    return filtered_grid_gdf

In [ ]:
def area_interpolate(grid_gdf, data_gdf, extensive_variables, intensive_variables, projected_crs):
    assert grid_gdf.crs == data_gdf.crs
    
    original_crs = grid_gdf.crs
    
    interpolated_gdf = tobler.area_weighted.area_interpolate(
        source_df=data_gdf.to_crs(projected_crs),
        target_df=grid_gdf.to_crs(projected_crs),
        extensive_variables=extensive_variables,
        intensive_variables=intensive_variables
    )

    return interpolated_gdf.to_crs(original_crs)

In [ ]:
BARCELONA_CRS = "EPSG:25831"



# Part 2

In [ ]:
def airbnb_listings(city_path, quarter_end_dates):
    """load listings for a city_path for a series of dates (representing quarter ends) and combines into one DataFrame.

    Adds a new `quarter_end_date` column so that data for each quarter can still be pulled out later.
    """
    listings_url_base = f"https://data.insideairbnb.com/{city_path}"
    dfs = []
    for quarter_end_date in quarter_end_dates:
        listings_url = f"{listings_url_base}/{quarter_end_date}/data/listings.csv.gz"
        df = pd.read_csv(listings_url, compression='gzip')
        df['quarter_end_date'] = quarter_end_date
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [ ]:
airbnb_df = airbnb_listings("spain/catalonia/barcelona", ["2024-12-12","2025-03-05","2025-06-12","2025-09-14"])
airbnb_df

In [ ]:
def create_gdf_from_latlon(df):
    """creates a new GDF from a Dataframe containing `latitude` and `longitude` columns."""
    geometry = gpd.points_from_xy(df['longitude'], df['latitude'], crs="EPSG:4326")
    return gpd.GeoDataFrame(df, geometry=geometry)

In [ ]:
airbnb_gdf = create_gdf_from_latlon(airbnb_df)
airbnb_gdf

In [ ]:
def price_only(gdf):
    """Takes a GDF with a `price` column formatted as a $100.00 string and turns it into a float `price` column.
    The `price` and geometry columns are returned in a new GDF.
    """
    price_gdf = gdf.copy(deep=True)
    price_gdf["price"] = (
        gdf["price"]
          .str.replace(r"[\$,]", "", regex=True)
          .astype(float)
    )
    price_gdf = price_gdf[["price", gdf.geometry.name]]
    price_gdf = price_gdf.dropna()
    return price_gdf

In [ ]:
airbnb_prices_gdf = price_only(airbnb_gdf)

In [ ]:
sns.displot(airbnb_prices_gdf["price"])

In [ ]:
sns.displot(filter_extremes(airbnb_prices_gdf, column="price"))

In [ ]:
def types_only(gdf):
    """Takes a GDF and finds all columns whose name contains `type`, and filters out any rows with no values or empty string.

    It keeps any columns that related to identity.
    """
    types_gdf = gdf.copy(deep=True)
    
    # find columns related to `type`
    columns = types_gdf.columns
    type_cols = list(columns[columns.str.contains("type")])
    keep_cols = type_cols + ["id",types_gdf.geometry.name]
    types_gdf = types_gdf[keep_cols]

    # remove any rows with empty strings or missing values
    for col in type_cols:
        types_gdf[col] = (
            types_gdf[col].str.replace(r"^\s*$", "", regex=True).replace("", None)
        )
    types_gdf = types_gdf.dropna()
    
    return types_gdf

In [ ]:
airbnb_types_gdf = types_only(airbnb_gdf)
airbnb_types_gdf


In [ ]:
sns.histplot(airbnb_types_gdf["property_type"])

In [ ]:
sns.histplot(airbnb_types_gdf["room_type"])

In [ ]:
airbnb_types_gdf.explore("room_type")

In [ ]:
airbnb_types_deduped_gdf = airbnb_types_gdf.drop_duplicates("geometry").copy()
airbnb_types_deduped_gdf.explore("room_type")

# Regionalisation notes

https://martinfleischmann.net/sds/micro/clustering/hands_on.html#spatially-constrained-clustering-regionalisation

1. Determine the number of Airbnb's per census area or grid area
2. Build an Agglomerative model between these areas based on count of airbnbs

Repeat and maybe play with cluster sizes. Also, I am not sure if agglomeration is based on a continuous distance or relies on categories. If latter, may need to bucket counts.

Maybe also want to, if have time, regard the prices and %-age of categories as other things to fit on.